# 02 Main Results — Rolling Spectral Estimation on S&P 100

This notebook runs the full pipeline on the 100-ticker S&P universe:

1. Load or download returns (2000–2023)
2. Run `RollingSpectralEstimator` (window=252, step=21)
3. Apply CUSUM change-point detection
4. Crisis event study and ROC validation
5. All six publication figures

---

**Prerequisites**: Run `download_returns()` once before executing this notebook,
or place a pre-built `research/data/returns_panel.parquet` in the data directory.

```python
from research.core.loader import download_returns
from research.core.universe import SP100_TICKERS
download_returns(SP100_TICKERS)  # downloads and caches to research/data/
```

---

**Limitations**:
- Survivorship bias: the SP100_TICKERS list includes only stocks with
  substantial presence in the S&P 500 during 2000–2023. Stocks that
  were removed from the index (due to delisting, acquisition, or
  decline) are underrepresented. Results are NOT representative of
  the full S&P 500 index.
- Within-window standardization: the estimator standardizes each asset
  to unit variance within each window (analyzing correlation structure,
  not covariance). Heterogeneous volatility across assets is removed.
- CUSUM calibration: all alarm times are sensitive to the choice of
  calibration period. We use the first 20 windows (approximately
  one year of monthly steps). This choice must be documented in any
  published results.
- Crisis calendar look-ahead: the event study labels are constructed
  after observing the data. AUROC values are descriptive, not predictive.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from research.core.universe import SP100_TICKERS
from research.core.loader import load_returns, returns_to_numpy
from research.core.estimator import RollingSpectralEstimator
from research.core.changepoint import cusum_from_snapshots
from research.analysis.validation import run_full_validation
from research.analysis.plots import all_figures
from research.core.mp_theory import verify_bbp_distinction

print(f'Universe: {len(SP100_TICKERS)} tickers')

## 1. Load Data

In [ ]:
# Load cached returns panel
# If FileNotFoundError: run download_returns(SP100_TICKERS) first (see Prerequisites above)
returns = load_returns()
print(f'Returns shape: {returns.shape}')
print(f'Date range: {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'Assets: {returns.shape[1]}')
print(f'Missing fraction: {returns.isna().mean().mean():.3f}')

In [ ]:
# Convert to numpy for the estimator
R, dates, tickers = returns_to_numpy(returns)
print(f'Array shape: {R.shape}')
print(f'Sample tickers: {tickers[:5]}')

## 2. Rolling Spectral Estimation

In [ ]:
est = RollingSpectralEstimator(
    window=252,    # ~1 trading year
    step=21,       # ~1 trading month
    min_assets=30, # require at least 30 active assets per window
)

snapshots = est.fit(R)

print(f'Total windows computed: {len(snapshots)}')
print(f'Skipped windows: {len(est.skipped_windows)}')
if len(snapshots) > 0:
    snap0 = snapshots[0]
    print(f'\nFirst snapshot:')
    print(f'  gamma={snap0.gamma:.4f}, sigma2={snap0.sigma2:.4f}')
    print(f'  lambda+={snap0.lambda_plus:.4f}, lambda1={snap0.lambda1:.4f}')
    print(f'  k={snap0.k}, rho={snap0.rho:.4f}, KS={snap0.ks:.4f}')

In [ ]:
# Summary statistics across all windows
import pandas as pd

summary = pd.DataFrame({
    'sigma2':  [s.sigma2 for s in snapshots],
    'lambda_plus': [s.lambda_plus for s in snapshots],
    'lambda1': [s.lambda1 for s in snapshots],
    'k':       [s.k for s in snapshots],
    'rho':     [s.rho for s in snapshots],
    'ks':      [s.ks for s in snapshots],
    'r_eff':   [s.r_eff for s in snapshots],
    'kappa':   [s.kappa for s in snapshots],
})

summary.describe().round(4)

## 3. CUSUM Change-Point Detection

In [ ]:
# calibration_n=20: first 20 windows (~1 year) as in-control reference
# k_delta=0.5, k_h=4.0: standard heuristic parameters
# [Warning: see changepoint.py module docstring — stationarity assumption violated]

cusum_result, center_positions = cusum_from_snapshots(
    snapshots,
    calibration_n=20,
    k_delta=0.5,
    k_h=4.0,
)

print(f'In-control mean (KS): {cusum_result.mu:.4f}')
print(f'In-control std (KS):  {cusum_result.sigma:.4f}')
print(f'Alarm threshold h:    {cusum_result.h:.4f}')
print(f'Number of alarms:     {len(cusum_result.alarms)}')

if len(cusum_result.alarms) > 0:
    alarm_dates = [dates[snapshots[i].center_pos] for i in cusum_result.alarms[:10]]
    print(f'First 10 alarm dates: {[str(d.date()) for d in alarm_dates]}')

## 4. Crisis Event Study and ROC Validation

In [ ]:
validation = run_full_validation(
    snapshots,
    dates=dates,
    expand_days=21,  # include 3 weeks around each episode
)

print(f'Crisis windows: {validation["labels"].sum()}')
print(f'Calm windows:   {(validation["labels"]==0).sum()}')
print()

print('Event study results:')
for name, res in validation['event_study'].items():
    print(f'  {name:4s}: crisis_mean={res.mean_crisis:.4f}, '
          f'calm_mean={res.mean_calm:.4f}, '
          f'ratio={res.ratio:.3f}, p={res.p_value:.4f}')

print()
print('ROC AUROC values:')
for name, roc in validation['roc'].items():
    print(f'  {name:4s}: AUROC={roc.auroc:.4f}')

## 5. All Six Publication Figures

In [ ]:
figs = all_figures(
    snapshots=snapshots,
    cusum_result=cusum_result,
    roc_results=validation['roc'],
    dates=dates,
)

for name, fig in figs.items():
    fig.savefig(f'../data/{name}_sp100.pdf', bbox_inches='tight')
    plt.figure(fig.number)
    plt.show()
    
print('All figures saved to research/data/')

## 6. Cross-Module Sanity Check: verify_bbp_distinction

Per the design specification, we run `verify_bbp_distinction` at the end of the
main results notebook to confirm the BBP/MP distinction (Mathematical Flag 1)
in the context of a representative γ from the actual data.

In [ ]:
# Use median gamma and sigma2 from the rolling estimation
median_gamma  = float(np.median([s.gamma  for s in snapshots]))
median_sigma2 = float(np.median([s.sigma2 for s in snapshots]))

print('Cross-module sanity check with data-derived parameters:')
print(f'  Median gamma from rolling windows: {median_gamma:.4f}')
print(f'  Median sigma2 from rolling windows: {median_sigma2:.4f}')
print()
verify_bbp_distinction(gamma=median_gamma, sigma2=median_sigma2)